<img src="../../figs/holberton_logo.png" alt="logo" width="500"/>


# Object Detection: YOLO Algorithm

## Key Goal of Object Detection

Object detection is a fundamental task in computer vision that involves **identifying and locating objects within an image or video**. The key goals of object detection are:

- **Localization**: Identifying the precise location of objects in the form of bounding boxes.
- **Classification**: Assigning a label or class to each detected object, indicating what type of object it is.

Object detection combines these two tasks to not only find objects but also recognize them. This has a wide range of applications including autonomous driving, surveillance, image search, medical imaging, and more

<img src="figs/od.png" alt="logo" width="700"/>


## YOLO Algorithm: You Only Look Once

YOLO (You Only Look Once) is a state-of-the-art, **real-time object detection system**. It is known for its **speed** and **accuracy**. The key idea behind YOLO is to *frame object detection as a single regression problem, straight from image pixels to bounding box coordinates and class probabilities*.

Here's a high-level overview of how YOLO works:

#### Single Neural Network 

Unlike traditional object detection systems that use a multi-stage pipeline, YOLO uses a single convolutional neural network (CNN) to predict multiple bounding boxes and class probabilities simultaneously.

#### Grid System

YOLO divides the input image into an $S x S$ grid. Each grid cell is responsible for predicting a fixed number of bounding boxes and the confidence scores for those boxes. Each bounding box comes with:

- Coordinates of the box (x, y, width, height)
- Confidence score indicating the probability of the box containing an object
- Class probabilities for each object category

#### Predicting Bounding Boxes and Classes

For each grid cell, YOLO predicts `B` bounding boxes and corresponding confidence scores. It also predicts `C` class probabilities. The final predictions are filtered based on confidence scores and non-max suppression to remove duplicate detections.

#### End-to-End Training

YOLO is trained end-to-end directly on full images and ground truth boxes. This unified architecture makes YOLO extremely fast and efficient.


## Advantages of YOLO:

- **Speed**: YOLO is significantly faster than other object detection algorithms, making it suitable for real-time applications.
- **Simplicity**: Its single-stage architecture simplifies the object detection pipeline, making it easier to implement and deploy.
- **Global Context**: YOLO reasons globally about the image when making predictions, which can improve detection accuracy for some objects.

## Tasks

### 0. Initialize Yolo

Write a class `Yolo` that uses the `Yolo v3` algorithm to perform object detection:

- class constructor: `def __init__(self, model_path, classes_path, class_t, nms_t, anchors)`:
    - `model_path` is the path to where a Darknet Keras model is stored
    - `classes_path` is the path to where the list of class names used for the Darknet model, listed in order of index, can be found
    - `class_t` is a float representing the box score threshold for the initial filtering step
    - `nms_t` is a float representing the IOU threshold for non-max suppression
    - `anchors` is a `numpy.ndarray` of shape (`outputs`, `anchor_boxes`, `2`) containing all of the anchor boxes:
        - `outputs` is the number of outputs (predictions) made by the Darknet model
        - `anchor_boxes` is the number of anchor boxes used for each prediction
        - `2 => [anchor_box_width, anchor_box_height]`
- Public instance attributes:
    - `model`: the Darknet Keras model
    - `class_names`: a list of the class names for the model
    - `class_t`: the box score threshold for the initial filtering step
    - `nms_t`: the IOU threshold for non-max suppression
    - `anchors`: the anchor boxes
    
    
### Solution Sketch

The `Yolo` class initializes a `YOLO v3` object detection system by performing the following steps:

- **Load Model**: Uses `load_model` to load a pre-trained YOLO v3 model from the specified `model_path`.
- **Load Class Names**: Reads the class names from `classes_path`, storing each name in a list `self.class_names`.
- **Set Thresholds**: Assigns the box score threshold (`class_t`) and the non-max suppression threshold (`nms_t`) to corresponding instance attributes.
- **Store Anchors**: Saves the provided anchor boxes (`anchors`) which are essential for predicting bounding box coordinates.

<img src="figs/acnhor.jpg" alt="logo" width="700"/>


#### Thresholds

The box score threshold (`class_t`) **determines the minimum confidence level required for a predicted bounding box to be considered valid**, filtering out low-confidence detections. The non-max suppression threshold (`nms_t`) sets the `Intersection Over Union (IOU)` value at which overlapping bounding boxes are considered redundant, ensuring only the most accurate bounding boxes for detected objects are kept.

#### Store Anchors

Anchor boxes (`anchors`) are predefined bounding box shapes and sizes used by the YOLO model to predict object locations. They help the model to detect objects of varying dimensions by providing reference points for the network's predictions, which are then adjusted to fit the actual objects detected in the image.

In [1]:
#!/usr/bin/env python3
"""Task 0. Initialize Yolo"""
from tensorflow.keras.models import load_model


class Yolo:
    
    def __init__(self, model_path, classes_path, class_t, nms_t, anchors):
        self.model = load_model(model_path)
        with open(classes_path) as f:
            self.class_names = [line.strip() for line in f]
        self.class_t = class_t
        self.nms_t = nms_t
        self.anchors = anchors

In [2]:
if __name__ == '__main__':
    import numpy as np
    
    np.random.seed(0)
    anchors = np.array([[[116, 90], [156, 198], [373, 326]],
                        [[30, 61], [62, 45], [59, 119]],
                        [[10, 13], [16, 30], [33, 23]]])
    yolo = Yolo('yolo.h5', 'coco_classes.txt', 0.6, 0.5, anchors)
    yolo.model.summary()
    print('Class names:', yolo.class_names)
    print('Class threshold:', yolo.class_t)
    print('NMS threshold:', yolo.nms_t)
    print('Anchor boxes:', yolo.anchors)

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 416, 416, 3)]        0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 416, 416, 32)         864       ['input_1[0][0]']             
                                                                                                  
 batch_normalization (Batch  (None, 416, 416, 32)         128       ['conv2d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 leaky_re_lu (LeakyReLU)     (None, 416, 416, 32)         0         ['batch_normalization[0][0

                                                                    ]                             
                                                                                                  
 conv2d_8 (Conv2D)           (None, 104, 104, 128)        73728     ['leaky_re_lu_7[0][0]']       
                                                                                                  
 batch_normalization_8 (Bat  (None, 104, 104, 128)        512       ['conv2d_8[0][0]']            
 chNormalization)                                                                                 
                                                                                                  
 leaky_re_lu_8 (LeakyReLU)   (None, 104, 104, 128)        0         ['batch_normalization_8[0][0]'
                                                                    ]                             
                                                                                                  
 add_2 (Ad

                                                                                                  
 add_5 (Add)                 (None, 52, 52, 256)          0         ['activation_4[0][0]',        
                                                                     'leaky_re_lu_15[0][0]']      
                                                                                                  
 activation_5 (Activation)   (None, 52, 52, 256)          0         ['add_5[0][0]']               
                                                                                                  
 conv2d_16 (Conv2D)          (None, 52, 52, 128)          32768     ['activation_5[0][0]']        
                                                                                                  
 batch_normalization_16 (Ba  (None, 52, 52, 128)          512       ['conv2d_16[0][0]']           
 tchNormalization)                                                                                
          

                                                                    ']                            
                                                                                                  
 add_9 (Add)                 (None, 52, 52, 256)          0         ['activation_8[0][0]',        
                                                                     'leaky_re_lu_23[0][0]']      
                                                                                                  
 activation_9 (Activation)   (None, 52, 52, 256)          0         ['add_9[0][0]']               
                                                                                                  
 conv2d_24 (Conv2D)          (None, 52, 52, 128)          32768     ['activation_9[0][0]']        
                                                                                                  
 batch_normalization_24 (Ba  (None, 52, 52, 128)          512       ['conv2d_24[0][0]']           
 tchNormal

 batch_normalization_31 (Ba  (None, 26, 26, 256)          1024      ['conv2d_31[0][0]']           
 tchNormalization)                                                                                
                                                                                                  
 leaky_re_lu_31 (LeakyReLU)  (None, 26, 26, 256)          0         ['batch_normalization_31[0][0]
                                                                    ']                            
                                                                                                  
 conv2d_32 (Conv2D)          (None, 26, 26, 512)          1179648   ['leaky_re_lu_31[0][0]']      
                                                                                                  
 batch_normalization_32 (Ba  (None, 26, 26, 512)          2048      ['conv2d_32[0][0]']           
 tchNormalization)                                                                                
          

                                                                                                  
 batch_normalization_39 (Ba  (None, 26, 26, 256)          1024      ['conv2d_39[0][0]']           
 tchNormalization)                                                                                
                                                                                                  
 leaky_re_lu_39 (LeakyReLU)  (None, 26, 26, 256)          0         ['batch_normalization_39[0][0]
                                                                    ']                            
                                                                                                  
 conv2d_40 (Conv2D)          (None, 26, 26, 512)          1179648   ['leaky_re_lu_39[0][0]']      
                                                                                                  
 batch_normalization_40 (Ba  (None, 26, 26, 512)          2048      ['conv2d_40[0][0]']           
 tchNormal

 batch_normalization_47 (Ba  (None, 13, 13, 1024)         4096      ['conv2d_47[0][0]']           
 tchNormalization)                                                                                
                                                                                                  
 leaky_re_lu_47 (LeakyReLU)  (None, 13, 13, 1024)         0         ['batch_normalization_47[0][0]
                                                                    ']                            
                                                                                                  
 add_20 (Add)                (None, 13, 13, 1024)         0         ['activation_19[0][0]',       
                                                                     'leaky_re_lu_47[0][0]']      
                                                                                                  
 activation_20 (Activation)  (None, 13, 13, 1024)         0         ['add_20[0][0]']              
          

                                                                    ']                            
                                                                                                  
 conv2d_56 (Conv2D)          (None, 13, 13, 512)          524288    ['leaky_re_lu_55[0][0]']      
                                                                                                  
 batch_normalization_56 (Ba  (None, 13, 13, 512)          2048      ['conv2d_56[0][0]']           
 tchNormalization)                                                                                
                                                                                                  
 leaky_re_lu_56 (LeakyReLU)  (None, 13, 13, 512)          0         ['batch_normalization_56[0][0]
                                                                    ']                            
                                                                                                  
 conv2d_59

 leaky_re_lu_66 (LeakyReLU)  (None, 52, 52, 128)          0         ['batch_normalization_66[0][0]
                                                                    ']                            
                                                                                                  
 conv2d_69 (Conv2D)          (None, 52, 52, 256)          294912    ['leaky_re_lu_66[0][0]']      
                                                                                                  
 batch_normalization_67 (Ba  (None, 52, 52, 256)          1024      ['conv2d_69[0][0]']           
 tchNormalization)                                                                                
                                                                                                  
 leaky_re_lu_67 (LeakyReLU)  (None, 52, 52, 256)          0         ['batch_normalization_67[0][0]
                                                                    ']                            
          

## 4. Load images

Add the static method `def load_images(folder_path)`:
- `folder_path`: a string representing the path to the folder holding all the images to load
- Returns a tuple of (`images, image_paths`):
    - `images`: a list of images as `numpy.ndarrays`
    - `image_paths`: a list of paths to the individual images in images
    
### Key concepts

This function is designed to **load and return a list of images and their corresponding file paths from a specified folder**. It leverages the `glob` module to find all files matching a pattern (in this case, all files in the folder), and `OpenCV` (`cv2.imread`) to read each file as an image object.

- `@staticmethod`: Indicates that load_images is a static method, meaning it doesn't require an instance of the class to be called.
- Uses the glob module to find all files (images) within the specified `folder_path`. 
- Reads each image file identified by `image_paths` using `OpenCV`'s imread function, creating a list of image objects (images).

In [3]:
#!/usr/bin/env python3
"""Yolo"""
from tensorflow.keras.models import load_model
import numpy as np
import glob
import cv2


class Yolo:
    
    def __init__(self, model_path, classes_path, class_t, nms_t, anchors):
        self.model = load_model(model_path)
        with open(classes_path) as f:
            self.class_names = [line.strip() for line in f]
        self.class_t = class_t
        self.nms_t = nms_t
        self.anchors = anchors
        
    @staticmethod
    def load_images(folder_path):
        """Loads images from a filepath
        """
        image_paths = glob.glob(folder_path + '/*')
        images = [cv2.imread(image) for image in image_paths]
        return images, image_paths
        
    

In [4]:
import cv2
import numpy as np

np.random.seed(2)
anchors = np.array([[[116, 90], [156, 198], [373, 326]],
                    [[30, 61], [62, 45], [59, 119]],
                    [[10, 13], [16, 30], [33, 23]]])
yolo = Yolo('yolo.h5', 'coco_classes.txt', 0.6, 0.5, anchors)
images, image_paths = yolo.load_images('yolo_images/yolo/')
image_paths, images = zip(*sorted(zip(image_paths, images)))
i = np.random.randint(0, len(images))
print(i)
print(image_paths[i])
cv2.imshow(image_paths[i], images[i])
cv2.waitKey(0)
cv2.destroyAllWindows()

8
yolo_images/yolo\takagaki.jpg


## 1. Process Outputs

Add the public method `def process_outputs(self, outputs, image_size)`:
- `outputs` is a list of `numpy.ndarrays` containing the predictions from the Darknet model for a single image:
    - Each output will have the shape (`grid_height`, `grid_width`, `anchor_boxes`, `4 + 1 + classes`)
        - `grid_height` & `grid_width` => the height and width of the grid used for the output
        - `anchor_boxes` => the number of anchor boxes used
        - `4 => (t_x, t_y, t_w, t_h)`
        - `1 => box_confidence`
        - `classes` => class probabilities for all classes
    - `image_size` is a `numpy.ndarray` containing the image’s original size `[image_height, image_width]`
    - Returns a tuple of (`boxes`, `box_confidences`, `box_class_probs`):
        - `boxes`: a list of `numpy.ndarrays` of shape (`grid_height, grid_width, anchor_boxes, 4`) containing the processed boundary boxes for each output, respectively:
            - `4 => (x1, y1, x2, y2)`
            - `(x1, y1, x2, y2)` should represent the boundary box relative to original image
        - `box_confidences`: a list of `numpy.ndarrays` of shape `(grid_height, grid_width, anchor_boxes, 1)` containing the box confidences for each output, respectively
        - `box_class_probs`: a list of `numpy.ndarrays` of shape (`grid_height, grid_width, anchor_boxes, classes`) containing the box’s class probabilities for each output, respectively
        
### High-Level Explanation of Method `process_outputs`

The `process_outputs` method aims to **process the raw predictions from the YOLO model into a more interpretable format**. This involves *extracting and transforming the bounding box coordinates, confidence scores, and class probabilities* so that they can be used to identify and locate objects within the original image.

### Key Steps

#### 1. Initialization

Three lists are initialized: `boxes`, `box_confidences`, and `box_class_probs` to store the processed bounding boxes, their confidence scores, and class probabilities respectively.

#### 2. Extract raw predictions

- For each output from the YOLO model, the method extracts:
    - The raw bounding box predictions (`tx`, `ty`, `tw`, `th`).
    - The box confidence scores using a sigmoid function on the fifth element of the output array.
    - The class probabilities using a sigmoid function on the remaining elements.
    
#### 3. Adjust Bounding Boxes:

- The method iterates through each grid cell and anchor box in the outputs:
    - It decodes the raw bounding box predictions using the sigmoid function to get the coordinates relative to the grid cell (`bx`, `by`) and scales (`bw`, `bh`) using exponential functions and anchor dimensions.
    - The decoded coordinates are then scaled to the original image size and transformed to represent the corners of the bounding box (`x1`, `y1`, `x2`, `y2`).
    
#### 4. Normalization and Scaling:

- The bounding box coordinates are normalized by the grid width and height and then scaled to match the original image dimensions.
- This ensures that the bounding boxes are correctly mapped onto the original image

#### 5. Return Processed Outputs:

- The method returns the lists of processed bounding boxes, box confidence scores, and class probabilities, which can then be used for further filtering and object detection

In [ ]:
#!/usr/bin/env python3
"""Task 0. Initialize Yolo"""
from tensorflow.keras.models import load_model
import numpy as np


class Yolo:
   
    def __init__(self, model_path, classes_path, class_t, nms_t, anchors):
        self.model = load_model(model_path)
        with open(classes_path) as f:
            self.class_names = [line.strip() for line in f]
        self.class_t = class_t
        self.nms_t = nms_t
        self.anchors = anchors

    def process_outputs(self, outputs, image_size):
        
        boxes = []
        box_confidences = []
        box_class_probs = []
        
        for i in range(len(outputs)):
            # Extract bounding boxes, confidence scores, and class probabilities
            boxes.append(outputs[i][..., :4])
            box_confidences.append(1 / (1 + np.exp(-outputs[i][..., 4:5])))
            box_class_probs.append(1 / (1 + np.exp(-outputs[i][..., 5:])))
        
        image_height, image_width = image_size
        
        for i in range(len(boxes)):
            
            grid_width = outputs[i].shape[1]
            grid_height = outputs[i].shape[0]
            anchor_boxes = outputs[i].shape[2]
            
            for cy in range(grid_height):
                for cx in range(grid_width):
                    for b in range(anchor_boxes):
                        
                        # Decode bounding box predictions
                        tx, ty, tw, th = boxes[i][cy, cx, b]
                        pw, ph = self.anchors[i][b]
                        
                        bx = (1 / (1 + np.exp(-tx))) + cx
                        by = (1 / (1 + np.exp(-ty))) + cy
                        
                        bw = pw * np.exp(tw)
                        bh = ph * np.exp(th)
                        
                        # Normalize coordinates
                        bx /= grid_width
                        by /= grid_height
                        
                        bw /= self.model.input.shape[1]
                        bh /= self.model.input.shape[2]
                        
                        # Scale to original image size
                        x1 = (bx - (bw / 2)) * image_width
                        y1 = (by - (bh / 2)) * image_height
                        x2 = (bx + (bw / 2)) * image_width
                        y2 = (by + (bh / 2)) * image_height
                        
                        boxes[i][cy, cx, b] = [x1, y1, x2, y2]
        
        return (boxes, box_confidences, box_class_probs)

In [ ]:
#!/usr/bin/env python3

if __name__ == '__main__':
    import numpy as np

    np.random.seed(0)
    anchors = np.array([[[116, 90], [156, 198], [373, 326]],
                        [[30, 61], [62, 45], [59, 119]],
                        [[10, 13], [16, 30], [33, 23]]])
    yolo = Yolo('yolo.h5', 'coco_classes.txt', 0.6, 0.5, anchors)
    output1 = np.random.randn(13, 13, 3, 85)
    output2 = np.random.randn(26, 26, 3, 85)
    output3 = np.random.randn(52, 52, 3, 85)
    boxes, box_confidences, box_class_probs = yolo.process_outputs([output1, output2, output3], np.array([500, 700]))
    print('Boxes:', boxes)
    print('Box confidences:', box_confidences)
    print('Box class probabilities:', box_class_probs)

## 2. Filter Boxes

Add the public method `def filter_boxes(self, boxes, box_confidences, box_class_probs)`:
- `boxes`: a list of `numpy.ndarrays` of shape (`grid_height`, `grid_width`, `anchor_boxes`, `4`) containing the processed boundary boxes for each output, respectively
- `box_confidences`: a list of `numpy.ndarrays` of shape (`grid_height`, `grid_width`, `anchor_boxes`, `1`) containing the processed box confidences for each output, respectively
- `box_class_probs`: a list of `numpy.ndarrays` of shape (`grid_height`, `grid_width`, `anchor_boxes`, `classes`) containing the processed box class probabilities for each output, respectively
- Returns a tuple of (`filtered_boxes, box_classes, box_scores`):
    - `filtered_boxes`: a `numpy.ndarray` of shape `(?, 4)` containing all of the filtered bounding boxes:
    - `box_classes`: a `numpy.ndarray` of shape `(?,)` containing the class number that each box in filtered_boxes predicts, respectively
    - `box_scores`: a `numpy.ndarray` of shape `(?)` containing the box scores for each box in `filtered_boxes`, respectively
    
### High Level Explanation of Method `filter_boxes`

The `filter_boxes` method aims to **filter out low-confidence bounding boxes from the YOLO model's predictions**, retaining only those boxes that meet a specified confidence threshold. This method processes the bounding boxes, confidence scores, and class probabilities, returning only the high-confidence boxes along with their associated class labels and scores.

### Key Steps

#### 1. Initialization:

- Initialize `filtered_boxes` as `None` and `box_classes` and `box_scores` as empty lists. These will store the final filtered results.


#### 2. Compute Box Scores:

- For each output, calculate the box scores by multiplying the box confidences with the class probabilities. This gives a score for each class for each bounding box.
- Determine the class with the highest score (`cur_box_class`) and the highest score value (`cur_box_score`) for each bounding box.

#### 3. Apply Confidence Threshold:

- Create a mask to filter out boxes with scores below the specified `class_t` threshold.


#### 4. Filter and Collect Boxes:

- Apply the mask to extract the high-confidence boxes, their scores, and corresponding class labels.
- If filtered_boxes is None, initialize it with the filtered boxes from the first output. Otherwise, concatenate the new filtered boxes with the existing ones.

#### 5. Return Filtered Results:

Return the `filtered_boxes`, `box_classes`, and `box_scores` as numpy arrays, representing the bounding boxes, their predicted classes, and their confidence scores respectively.

In [ ]:
#!/usr/bin/env python3
"""Task 0. Initialize Yolo"""
from tensorflow.keras.models import load_model
import numpy as np


class Yolo:
   
    def __init__(self, model_path, classes_path, class_t, nms_t, anchors):
        self.model = load_model(model_path)
        with open(classes_path) as f:
            self.class_names = [line.strip() for line in f]
        self.class_t = class_t
        self.nms_t = nms_t
        self.anchors = anchors

    def process_outputs(self, outputs, image_size):
        
        boxes = []
        box_confidences = []
        box_class_probs = []
        
        for i in range(len(outputs)):
            # Extract bounding boxes, confidence scores, and class probabilities
            boxes.append(outputs[i][..., :4])
            box_confidences.append(1 / (1 + np.exp(-outputs[i][..., 4:5])))
            box_class_probs.append(1 / (1 + np.exp(-outputs[i][..., 5:])))
        
        image_height, image_width = image_size
        
        for i in range(len(boxes)):
            
            grid_width = outputs[i].shape[1]
            grid_height = outputs[i].shape[0]
            anchor_boxes = outputs[i].shape[2]
            
            for cy in range(grid_height):
                for cx in range(grid_width):
                    for b in range(anchor_boxes):
                        
                        # Decode bounding box predictions
                        tx, ty, tw, th = boxes[i][cy, cx, b]
                        pw, ph = self.anchors[i][b]
                        
                        bx = (1 / (1 + np.exp(-tx))) + cx
                        by = (1 / (1 + np.exp(-ty))) + cy
                        
                        bw = pw * np.exp(tw)
                        bh = ph * np.exp(th)
                        
                        # Normalize coordinates
                        bx /= grid_width
                        by /= grid_height
                        
                        bw /= self.model.input.shape[1]
                        bh /= self.model.input.shape[2]
                        
                        # Scale to original image size
                        x1 = (bx - (bw / 2)) * image_width
                        y1 = (by - (bh / 2)) * image_height
                        x2 = (bx + (bw / 2)) * image_width
                        y2 = (by + (bh / 2)) * image_height
                        
                        boxes[i][cy, cx, b] = [x1, y1, x2, y2]
        
        return (boxes, box_confidences, box_class_probs)
    
    def filter_boxes(self, boxes, box_confidences, box_class_probs):
        """Removes the boxes with low box scores"""
        
        filtered_boxes, box_classes, box_scores = None, [], []
        
        for i in range(len(boxes)):
            # Compute the box scores
            cur_box_score = box_confidences[i] * box_class_probs[i]
            cur_box_class = np.argmax(cur_box_score, axis=-1)
            cur_box_score = np.max(cur_box_score, axis=-1)
            
            # Apply the confidence threshold
            mask = cur_box_score >= self.class_t
            
            # Filter and collect boxes
            if filtered_boxes is None:
                
                filtered_boxes = boxes[i][mask]
                box_scores = cur_box_score[mask]
                box_classes = cur_box_class[mask]
            
            else:
                
                filtered_boxes = np.concatenate((filtered_boxes, boxes[i][mask]), axis=0)
                box_classes = np.concatenate((box_classes, cur_box_class[mask]), axis=0)
                box_scores = np.concatenate((box_scores, cur_box_score[mask]), axis=0)
                
        return (filtered_boxes, box_classes, box_scores)

In [ ]:
if __name__ == '__main__':
    import numpy as np

    np.random.seed(0)
    anchors = np.array([[[116, 90], [156, 198], [373, 326]],
                        [[30, 61], [62, 45], [59, 119]],
                        [[10, 13], [16, 30], [33, 23]]])
    yolo = Yolo('yolo.h5', 'coco_classes.txt', 0.6, 0.5, anchors)
    output1 = np.random.randn(13, 13, 3, 85)
    output2 = np.random.randn(26, 26, 3, 85)
    output3 = np.random.randn(52, 52, 3, 85)
    boxes, box_confidences, box_class_probs = yolo.process_outputs([output1, output2, output3], np.array([500, 700]))
    boxes, box_classes, box_scores = yolo.filter_boxes(boxes, box_confidences, box_class_probs)
    print('Boxes:', boxes)
    print('Box classes:', box_classes)
    print('Box scores:', box_scores)